# 17 · Landkarten mit geopandas

**Ausblick:** Dieses Notebook ist eine Vorführung ohne Übungen.

**Lernziele**

- Du weißt, was Geodaten sind: eine Tabelle mit einer Spalte für die Form.
- Du kennst die Grundformen Punkt, Linie und Fläche und weißt, warum es verschiedene Koordinatensysteme gibt.
- Du färbst die Landkreise Deutschlands nach einer Kennzahl ein.
- Du filterst eine Karte auf ein Bundesland und erstellst eine interaktive Karte, die beim Darüberfahren Werte zeigt.
- Du ordnest Orte automatisch dem Kreis zu, in dem sie liegen.

## Was ist geopandas?

In Notebook 15 hast du mit pandas Tabellen ausgewertet. **geopandas** erweitert pandas um eine Spalte, die eine Form enthält: den Umriss eines Landkreises, den Verlauf einer Straße, die Lage einer Filiale. Alles, was du aus pandas kennst, funktioniert weiter. Dazu kommt:

- Geodateien lesen und schreiben (GeoJSON, Shapefile, GeoPackage),
- Karten zeichnen, statisch oder interaktiv,
- mit Formen rechnen: Fläche, Entfernung, liegt ein Punkt in einer Fläche?
- Tabellen über die Lage verbinden statt über eine gemeinsame Spalte.

| Klasse | steht für | kennst du aus pandas als |
| --- | --- | --- |
| `GeoDataFrame` | Tabelle mit einer Spalte `geometry` | `DataFrame` |
| `GeoSeries` | die Spalte `geometry` selbst | `Series` |
| `Point`, `LineString`, `Polygon` | eine einzelne Form (Paket shapely) | ein einzelner Wert |

![Ein GeoDataFrame mit drei Zeilen: Schlüssel, Kreisname und eine orange markierte Spalte geometry mit Einträgen wie POLYGON ((8.56 49.84, …)). Ein Pfeil .plot() führt zu drei gezeichneten Umrissen: Lahn-Dill-Kreis, Frankfurt, Darmstadt. Darunter: Jede Zeile ist ein Kreis, geometry enthält seinen Umriss als Liste von Koordinaten.](../assets/17_geodataframe.png)

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import LineString, Point, Polygon

## Die Grundformen

Jede Form ist aus Koordinaten gebaut, also aus Paaren von Zahlen. Ein Punkt ist ein Paar, eine Linie eine Folge von Punkten, eine Fläche ein geschlossener Umriss. Welche Zahlen das sind, legt das **Koordinatensystem** fest (englisch *coordinate reference system*, kurz CRS):

![Drei Kästen: Punkt (ein Paar Koordinaten, z. B. eine Filiale) mit Point(8.68, 50.11), Linie (Punkte der Reihe nach verbunden, z. B. eine Straße) mit LineString, Fläche (geschlossener Umriss, z. B. ein Kreis) mit Polygon. Darunter zwei Koordinatensysteme: EPSG:4326 mit Länge und Breite in Grad, weltweit, zum Speichern und für Webkarten; EPSG:25832 mit x und y in Metern, für Deutschland, zum Rechnen und Zeichnen.](../assets/17_geometrien.png)

Bevor wir echte Karten laden, bauen wir ein kleines Werksgelände von Hand, in Metern: das Gelände als Fläche, ein Förderband als Linie, die Pforte als Punkt. Die Formen kommen in die Spalte `geometry`, die übrigen Spalten sind ganz normale pandas-Spalten:

In [ ]:
werk = gpd.GeoDataFrame(
    {"Name": ["Werksgelände", "Förderband", "Pforte"]},
    geometry=[
        Polygon([(0, 0), (60, 0), (60, 40), (0, 40)]),
        LineString([(10, 10), (30, 20), (50, 20)]),
        Point(45, 35),
    ],
)
werk

Jetzt rechnet geopandas mit den Formen, wie pandas mit Zahlen: für die ganze Spalte auf einmal. `area` ist die Fläche (hier m²), `length` die Länge einer Linie oder der Umfang einer Fläche. `contains()` prüft, ob eine Form eine andere enthält:

In [ ]:
print(werk.geom_type.tolist())
print(werk.area.tolist())
print(werk.length.round(1).tolist())

gelaende = werk.geometry[0]
pforte = werk.geometry[2]
print(gelaende.contains(pforte))
print(gelaende.contains(Point(70, 10)))

werk.plot(color=["lightgrey", "orange", "red"], edgecolor="black");

## Die Kreise Deutschlands

Die Datei `daten/kreise.geojson` enthält die Umrisse aller 400 Kreise und kreisfreien Städte, vereinfacht für den Maßstab 1:2 500 000. GeoJSON ist eine Textdatei im JSON-Format (Notebook 10); öffne sie ruhig einmal in VS Code.

**Woher die Daten kommen:** Das Bundesamt für Kartographie und Geodäsie (BKG) veröffentlicht die Verwaltungsgrenzen Deutschlands als freie Daten. Wir nutzen das Produkt *Verwaltungsgebiete 1:2 500 000* (VG2500), Stand 31.12.2022, eine ZIP-Datei mit Shapefiles:

```
https://daten.gdz.bkg.bund.de/produkte/vg/vg2500/2022/vg2500_12-31.utm32s.shape.zip
```

Das Skript `tools/make_karten_data.py` hat daraus `kreise.geojson` und `laender.geojson` gemacht und dabei verändert:

- nur Landflächen behalten, ohne Bodensee und Küstengewässer,
- nur die Spalten Schlüssel, Name und Art behalten und umbenannt,
- die Koordinaten von Metern (EPSG:25832) in Grad (EPSG:4326) umgerechnet, weil GeoJSON immer Grad speichert, und auf fünf Nachkommastellen (etwa 1 m) gerundet.

Die Daten stehen unter der *Datenlizenz Deutschland – Namensnennung – 2.0* (https://www.govdata.de/dl-de/by-2-0). Sie verlangt einen Quellenvermerk; wer die Daten bearbeitet, ergänzt „Daten verändert“. „GeoBasis-DE / BKG“ nennt die Herkunft: amtliche Geobasisdaten der Länder, bereitgestellt vom BKG. Die Vorgaben stehen unter https://gdz.bkg.bund.de/index.php/default/hinweise-zu-nutzungsbedingungen-und-quellenvermerken/.

*Quellenvermerk: © GeoBasis-DE / BKG (2023) (Daten verändert)*

`gpd.read_file()` liest die Datei, wie `pd.read_csv()` eine CSV-Datei:

In [ ]:
kreise = gpd.read_file("../daten/kreise.geojson")
print(type(kreise), len(kreise))
kreise.head()

Der Schlüssel ist der amtliche Regionalschlüssel: Die ersten zwei Ziffern sind das Bundesland, `06` steht für Hessen. Die Koordinaten sind Länge und Breite in Grad. Das verrät das Koordinatensystem `crs`:

In [ ]:
kreise.crs

In Grad kann geopandas zeichnen, aber nicht sinnvoll rechnen: Ein Grad Länge ist bei uns nur etwa 70 km breit, ein Grad Breite 111 km hoch. `area` liefert deshalb Werte in „Quadratgrad“, mit denen niemand etwas anfangen kann, und geopandas warnt:

In [ ]:
kreise.area.head(3)

`to_crs()` rechnet alle Koordinaten in ein anderes System um. EPSG:25832 misst in Metern und ist für Deutschland gemacht. `plot()` zeichnet die Umrisse; `set_axis_off()` blendet die Achsen aus, die Meterzahlen braucht auf einer Karte niemand:

In [ ]:
kreise = kreise.to_crs("EPSG:25832")

ax = kreise.plot(figsize=(6, 7), color="white", edgecolor="grey", linewidth=0.3)
ax.set_title("in Metern (EPSG:25832)")
ax.set_axis_off()

In Metern liefert `area` Quadratmeter. Geteilt durch eine Million ergibt das km², als neue Spalte wie in pandas:

In [ ]:
kreise["Flaeche_km2"] = (kreise.area / 1_000_000).round(1)
kreise[["Kreis", "Art", "Flaeche_km2"]].head()

## Einwohner dazuholen

Die Umrisse allein zeigen noch nichts. Die Einwohnerzahl je Kreis steht in einer zweiten Datei, einer CSV-Datei im deutschen Format wie in Notebook 15.

Achtung beim Schlüssel: pandas würde `06411` als Zahl lesen und die führende Null verlieren. `dtype={"Schluessel": str}` liest die Spalte als Text.

**Woher die Daten kommen:** Statistisches Bundesamt, Gemeindeverzeichnis, Tabelle *Kreisfreie Städte und Landkreise nach Fläche, Bevölkerung und Bevölkerungsdichte*, Stand 31.12.2024:

```
https://www.destatis.de/DE/Themen/Laender-Regionen/Regionales/Gemeindeverzeichnis/Administrativ/04-kreise.xlsx?__blob=publicationFile
```

`tools/make_karten_data.py` hat daraus vier Spalten übernommen (Schlüssel, Name, Fläche, Einwohner) und als CSV gespeichert. Die Tabelle erlaubt Vervielfältigung und Verbreitung mit Quellenangabe.

*Quellenvermerk: © Statistisches Bundesamt (Destatis), 2025*

In [ ]:
einwohner = pd.read_csv(
    "../daten/kreise_bevoelkerung.csv", sep=";", decimal=",", dtype={"Schluessel": str}
)
einwohner.head()

Die Namen passen nicht zusammen: `Flensburg` in der Karte, `Flensburg, Stadt` in der Statistik. Der Schlüssel ist dagegen in beiden Dateien gleich. Deshalb verbinden wir über den Schlüssel, mit `merge()` wie in Notebook 15. Für spätere Fälle zeigt die Grafik schon den zweiten Weg, den räumlichen Join; er folgt am Ende des Notebooks.

![Links merge(): eine Tabelle mit Schlüssel und geometry plus eine Tabelle mit Schlüssel und Einwohner ergibt eine Tabelle mit allen drei Spalten; gleicher Schlüssel, gleiche Zeile. Rechts sjoin(): die Punkte Wetzlar und Gießen liegen in den Flächen Lahn-Dill-Kreis und Gießen; daraus entsteht eine Tabelle Filiale und Kreis, ohne gemeinsame Spalte.](../assets/17_verbinden.png)

Beide Tabellen haben eine Spalte `Flaeche_km2`. `suffixes` hängt an die zweite `_amtlich` an, damit wir unsere berechnete Fläche mit der amtlichen vergleichen können:

In [ ]:
karte = kreise.merge(
    einwohner[["Schluessel", "Flaeche_km2", "Einwohner"]],
    on="Schluessel",
    suffixes=("", "_amtlich"),
)
karte["Dichte"] = (karte["Einwohner"] / karte["Flaeche_km2_amtlich"]).round()

print(type(karte), len(karte))
karte[["Kreis", "Flaeche_km2", "Flaeche_km2_amtlich", "Einwohner", "Dichte"]].head()

Das Ergebnis ist wieder ein `GeoDataFrame`, alle 400 Kreise haben einen Partner gefunden. Unsere Fläche weicht von der amtlichen ab, bei kleinen Städten wie Neumünster um mehr als zehn Prozent, weil die Umrisse vereinfacht sind. Für die Dichte (Einwohner je km²) nehmen wir deshalb die amtliche Fläche. Am dichtesten und am dünnsten besiedelt:

In [ ]:
columns = ["Kreis", "Art", "Einwohner", "Dichte"]
print(karte.sort_values("Dichte", ascending=False)[columns].head(5))
print()
print(karte.sort_values("Dichte")[columns].head(3))

## Kreise einfärben

`plot(column="Dichte")` färbt jeden Kreis nach seinem Wert ein, `cmap` wählt die Farben (hier von Gelb über Orange nach Rot). Links eine gleichmäßige Farbskala von 35 bis 4844: Weil wenige Großstädte so dicht besiedelt sind, bleibt fast das ganze Land hellgelb.

Rechts teilt `scheme="quantiles"` die Kreise in fünf gleich große Gruppen, jede bekommt eine Farbe. Jetzt werden Ballungsräume und ländliche Gegenden sichtbar. Die Grenzen der Bundesländer kommen aus einer zweiten Datei und werden mit `ax=` in dieselbe Zeichnung gelegt:

In [ ]:
laender = gpd.read_file("../daten/laender.geojson").to_crs("EPSG:25832")

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 8))
karte.plot(column="Dichte", cmap="YlOrRd", legend=True, ax=left)
left.set_title("gleichmäßige Farbskala")
left.set_axis_off()

karte.plot(
    column="Dichte",
    cmap="YlOrRd",
    scheme="quantiles",
    k=5,
    legend=True,
    legend_kwds={"title": "Einwohner je km²", "fmt": "{:.0f}", "loc": "upper left", "bbox_to_anchor": (1, 1)},
    ax=right,
)
laender.boundary.plot(ax=right, color="black", linewidth=0.6)
right.set_title("fünf gleich große Gruppen")
right.set_axis_off()

## Nur Hessen

Ein Bundesland auszuwählen ist ein ganz normaler pandas-Filter: Alle hessischen Schlüssel beginnen mit `06`. Die Namen setzen wir mit `annotate()` an einen Punkt im Inneren jedes Kreises (`representative_point()`), auf einem halb durchsichtigen weißen Kästchen (`bbox`), damit sie auch auf dunklen Flächen lesbar bleiben. Die fünf kreisfreien Städte sind zu klein für ihre Namen; sie bekommen erst in der interaktiven Karte eine Beschriftung.

In [ ]:
hessen = karte[karte["Schluessel"].str.startswith("06")]
print(len(hessen), "Kreise in Hessen")

ax = hessen.plot(
    column="Dichte",
    cmap="YlOrRd",
    scheme="quantiles",
    k=5,
    edgecolor="white",
    figsize=(10, 12),
    legend=True,
    legend_kwds={"title": "Einwohner je km²", "fmt": "{:.0f}", "loc": "upper left", "bbox_to_anchor": (1, 1)},
)
label_box = {"facecolor": "white", "alpha": 0.7, "edgecolor": "none", "pad": 1}
for _, row in hessen[hessen["Art"] == "Landkreis"].iterrows():
    point = row.geometry.representative_point()
    ax.annotate(row["Kreis"], (point.x, point.y), ha="center", fontsize=7, bbox=label_box)
ax.set_title("Hessen")
ax.set_axis_off()

## Interaktive Karte

`explore()` erstellt dieselbe Karte als Webseite: Du kannst zoomen und verschieben, und wenn du mit der Maus über einen Kreis fährst, zeigt ein Hinweisfeld (Tooltip) die Spalten aus `tooltip`. Im Hintergrund liegt eine OpenStreetMap-Karte; die lädt der Browser aus dem Internet. Die Koordinaten rechnet `explore()` selbst in Grad um.

`save()` speichert die Karte als HTML-Datei, die jeder Browser öffnet, auch ohne Python:

In [ ]:
interaktiv = hessen.explore(
    column="Dichte",
    cmap="YlOrRd",
    scheme="quantiles",
    k=5,
    tooltip=["Kreis", "Art", "Einwohner", "Dichte"],
    legend_kwds={"colorbar": False, "fmt": "{:.0f}", "caption": "Einwohner je km²"},
)

out_dir = Path("../daten/ausgabe")
out_dir.mkdir(exist_ok=True)
interaktiv.save(out_dir / "hessen_karte.html")
interaktiv

## Filialen auf der Karte

Die Filialen aus den Notebooks 15 und 16 stehen in `daten/filialen.csv` mit Länge und Breite ihrer Stadtmitte. `gpd.points_from_xy()` macht aus den beiden Zahlenspalten Punkte; `crs="EPSG:4326"` sagt, dass es Grad sind. Danach rechnen wir sie wie die Kreise in Meter um:

In [ ]:
filialen = pd.read_csv("../daten/filialen.csv", sep=";", decimal=",")
filialen = gpd.GeoDataFrame(
    filialen,
    geometry=gpd.points_from_xy(filialen["Laenge"], filialen["Breite"]),
    crs="EPSG:4326",
).to_crs("EPSG:25832")
filialen

Weil die Koordinaten jetzt Meter sind, liefert `distance()` Entfernungen in Metern, für alle Filialen auf einmal. Wie weit liegt jede Filiale von Frankfurt entfernt (Luftlinie)?

In [ ]:
frankfurt = filialen.loc[filialen["Filiale"] == "Frankfurt", "geometry"].iloc[0]
filialen["km_bis_Frankfurt"] = (filialen.distance(frankfurt) / 1000).round()
filialen[["Filiale", "km_bis_Frankfurt"]].sort_values("km_bis_Frankfurt")

In welchem Kreis liegt welche Filiale? Eine gemeinsame Spalte gibt es nicht; die Filialen-Tabelle kennt nur Koordinaten. `sjoin()` (*spatial join*, räumlicher Join) verbindet deshalb über die Lage: Jeder Punkt bekommt die Zeile der Fläche, in der er liegt (`predicate="within"`). Beide Tabellen müssen dafür im selben Koordinatensystem sein:

In [ ]:
zuordnung = filialen.sjoin(karte[["Kreis", "Einwohner", "geometry"]], predicate="within")
zuordnung[["Filiale", "Kreis", "Einwohner"]]

Wetzlar liegt im Lahn-Dill-Kreis, Siegen im Kreis Siegen-Wittgenstein, die übrigen Filialen in kreisfreien Städten gleichen Namens. `index_right` ist die Zeilennummer des Kreises in `karte`; damit holen wir die Kreise mit Filiale und zeichnen sie mit den Punkten in eine Karte von Nordrhein-Westfalen (`05`), Hessen (`06`) und Rheinland-Pfalz (`07`):

In [ ]:
region = karte[karte["Schluessel"].str[:2].isin(["05", "06", "07"])]
mit_filiale = karte.loc[zuordnung["index_right"]]

ax = region.plot(color="whitesmoke", edgecolor="grey", linewidth=0.3, figsize=(8, 9))
mit_filiale.plot(ax=ax, color="lightblue", edgecolor="grey")
laender[laender["Schluessel"].isin(["05", "06", "07"])].boundary.plot(ax=ax, color="black", linewidth=0.8)
filialen.plot(ax=ax, color="red", markersize=40, zorder=3)
for _, row in filialen.iterrows():
    # Wetzlar liegt dicht neben Gießen: Name links vom Punkt statt rechts
    if row["Filiale"] == "Wetzlar":
        ax.annotate(row["Filiale"], (row.geometry.x, row.geometry.y), xytext=(-6, 4), textcoords="offset points", ha="right")
    else:
        ax.annotate(row["Filiale"], (row.geometry.x, row.geometry.y), xytext=(6, 4), textcoords="offset points")
ax.set_axis_off()

## Zusammenfassung

- Ein `GeoDataFrame` ist ein `DataFrame` mit einer Spalte `geometry`. Filtern, Sortieren, neue Spalten und `merge()` funktionieren wie in pandas.
- Formen sind Punkte, Linien und Flächen (`Point`, `LineString`, `Polygon`), gebaut aus Koordinaten.
- Das Koordinatensystem (`crs`) sagt, was die Zahlen bedeuten. Grad (EPSG:4326) zum Speichern, Meter (EPSG:25832) zum Rechnen und Zeichnen; `to_crs()` rechnet um.
- `gpd.read_file()` liest Geodateien, `points_from_xy()` macht aus zwei Zahlenspalten Punkte.
- `area`, `length` und `distance()` rechnen mit den Formen, in Metern, wenn das CRS in Metern misst.
- `plot(column=...)` färbt nach einem Wert ein; `scheme="quantiles"` hilft bei wenigen extremen Werten. `explore()` macht daraus eine interaktive Karte mit Tooltip.
- `merge()` verbindet über einen Schlüssel, `sjoin()` über die Lage.

**Quellen der Daten** (Download-Adressen und Änderungen oben bei den Daten):

- Umrisse: © GeoBasis-DE / BKG (2023) (Daten verändert), VG2500 Stand 31.12.2022, Datenlizenz Deutschland – Namensnennung – 2.0.
- Fläche und Einwohner: © Statistisches Bundesamt (Destatis), 2025, Gemeindeverzeichnis, Stand 31.12.2024.